# 02 — Cleaning & building the analysis dataset

EDA gave us **3,288 paired funds**. This notebook turns raw NAV history into one
tidy, analysis-ready table — `data/processed/fund_returns.csv` — with one row per
fund and its measured commission drag.

Cleaning is where a fair analysis is won or lost, so **every decision below is
stated with its reason**. The guiding principle: *when in doubt, drop it rather
than risk a contaminated comparison.* We'd rather report an honest number on
fewer funds than an impressive number on dirty data.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import download, cleaning, plotting, paths
plotting.set_style()

## Step 1 — Rebuild the matched pairs

We regenerate the pairs from the raw scheme list (deterministic) rather than
trusting a stale file.

In [2]:
raw_list = download.fetch_scheme_list()
pairs = cleaning.build_pairs(raw_list)
print(f"{len(pairs):,} candidate Direct<->Regular Growth pairs")

3,288 candidate Direct<->Regular Growth pairs


## Step 2 — Compute the drag for every pair

For each fund we:

1. load both NAV series (dropping non-positive NAVs and duplicate dates),
2. find the **common date window** — both plans only co-exist from the later of
   their two start dates (Direct plans began in Jan 2013, so this is usually the
   binding constraint),
3. compute each plan's **CAGR** over that identical window, and
4. take `drag_pp = direct_cagr − regular_cagr` (percentage points per year).

`compute_pair_metrics` also enforces a **minimum 2-year overlap** — over very
short windows the two NAVs can differ by rounding/tracking noise that swamps the
true fee, so short-lived funds are excluded up front.

In [3]:
records = []
missing = 0
for row in pairs.itertuples(index=False):
    d_pay = download.fetch_scheme_nav(row.direct_code)
    r_pay = download.fetch_scheme_nav(row.regular_code)
    if d_pay is None or r_pay is None:
        missing += 1
        continue
    m = cleaning.compute_pair_metrics(d_pay, r_pay, min_years=2.0)
    if m is None:
        continue
    m["fund_name"] = row.fund_name
    m["direct_code"] = row.direct_code
    m["regular_code"] = row.regular_code
    records.append(m)

df = pd.DataFrame(records)
print(f"{len(df):,} funds with a valid >=2yr Direct/Regular comparison")
print(f"({missing} pairs skipped for missing NAV data, "
      f"{len(pairs) - len(df) - missing} for <2yr overlap)")
df.head(3)

2,312 funds with a valid >=2yr Direct/Regular comparison
(0 pairs skipped for missing NAV data, 976 for <2yr overlap)


,fund_house,scheme_type,scheme_category,window_start,window_end,window_years,direct_cagr,regular_cagr,drag_pp,fund_name,direct_code,regular_code
0,360 ONE Mutual Fund,Open Ended Schemes,Hybrid Scheme - Balanced Hybrid Fund,2023-10-03,2026-06-05,2.67,0.110205,0.093732,1.647301,360 ONE Balanced Hybrid Fund,152076,152075
1,360 ONE Mutual Fund,Open Ended Schemes,Debt Scheme - Dynamic Bond,2013-06-25,2026-06-05,12.94,0.073262,0.068642,0.461967,360 ONE Dynamic Bond Fund,122715,122612
2,360 ONE Mutual Fund,Open Ended Schemes,Equity Scheme - ELSS,2023-01-02,2026-06-05,3.42,0.084225,0.081516,0.270835,360 ONE ELSS Tax Saver Nifty 50 Index Fund,151165,151164


## Step 3 — Keep open-ended funds only

The pairs include **Fixed Maturity Plans (FMPs)** and other close-ended "series"
products — short-lived, you can't actually buy-and-hold them, and they pollute a
study about the funds ordinary investors sit in for years. AMFI's `scheme_type`
metadata lets us filter to genuine open-ended funds rather than guessing from names.

In [4]:
print("scheme_type values present:")
print(df["scheme_type"].value_counts(dropna=False), "\n")

before = len(df)
df = df[df["scheme_type"].str.contains("Open Ended", case=False, na=False)].copy()
print(f"Dropped {before - len(df):,} close-ended / interval schemes; {len(df):,} open-ended funds remain")

scheme_type values present:
scheme_type
Open Ended Schemes                                                                                   1084
Close Ended Schemes                                                                                   703
IL&FS Mutual Fund                                                                                     441
110594;#N/A;-;CANARA ROBECO FIXED MATURITY PLAN- Series 4                                              17
112659;INF090I01403;-;Franklin India Fixed Tenure Fund - Series XII - Plan C                           12
360 ONE Mutual Fund                                                                                     8
144748;INF209KB1VS8;-;Aditya Birla Sun Life Fixed Term Plan - Series QY                                 6
138655;INF663L01SO6;-;DHFL Pramerica Hybrid Fixed Term Fund - Series 19 - Regular Plan - Dividend       4
141581;INF179KB1V39;-;HDFC EOF - II - 1100D June 2017                                                   4
134059

## Step 4 — Add a broad category and sanity-check coverage

AMFI's `scheme_category` is granular (e.g. *Equity Scheme - Flexi Cap Fund*). We
add a coarse bucket — **Equity / Debt / Hybrid / Other-Index** — for grouping,
while keeping the detailed field.

In [5]:
df["broad_category"] = df["scheme_category"].map(cleaning.categorize_broad)
print("Funds per broad category:")
print(df["broad_category"].value_counts(), "\n")
print("Coverage: ", df["fund_house"].nunique(), "fund houses,",
      f"windows from {df['window_start'].min()} to {df['window_end'].max()}")

Funds per broad category:
broad_category
Equity         367
Other/Index    311
Debt           258
Hybrid         148
Name: count, dtype: int64 

Coverage:  50 fund houses, windows from 2013-01-01 to 2026-06-07


## Step 5 — Investigate outliers in the drag

We expect drag to be **small and positive**: a few tenths of a percent up to ~1.5
pp/yr (the commission). Let's look at the distribution and, critically, the tails.
A *negative* drag (Regular beat Direct) is economically impossible over a long
window for an identical portfolio, so any such cases flag data problems, not real
effects.

In [6]:
print(df["drag_pp"].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(3))

n_negative = (df["drag_pp"] < 0).sum()
n_extreme = (df["drag_pp"].abs() > 4).sum()
print(f"\nNegative drag (Regular 'beat' Direct): {n_negative} funds "
      f"({n_negative/len(df)*100:.1f}%)")
print(f"Extreme |drag| > 4 pp/yr: {n_extreme} funds")

print("\nMost negative cases (likely data issues / very short effective history):")
print(df.nsmallest(5, "drag_pp")[["fund_name","window_years","direct_cagr","regular_cagr","drag_pp"]]
      .to_string(index=False))
print("\nMost extreme positive cases:")
print(df.nlargest(5, "drag_pp")[["fund_name","window_years","direct_cagr","regular_cagr","drag_pp"]]
      .to_string(index=False))

count    1084.000
mean        0.888
std         0.985
min       -25.332
1%          0.002
5%          0.104
25%         0.461
50%         0.810
75%         1.383
95%         1.902
99%         2.185
max         2.517
Name: drag_pp, dtype: float64

Negative drag (Regular 'beat' Direct): 5 funds (0.5%)
Extreme |drag| > 4 pp/yr: 1 funds

Most negative cases (likely data issues / very short effective history):
                                                                  fund_name  window_years  direct_cagr  regular_cagr    drag_pp
Aditya Birla Sun Life Government Securities Fund - Short Term Plan - Growth          2.94    -0.176047      0.077277 -25.332428
                                                         Taurus Liquid Fund          7.84     0.062296      0.064653  -0.235704
                                               Tata Silver ETF Fund of Fund          2.36     0.665324      0.666428  -0.110454
                                   Tata Floating Rate Fund - Long Term Plan    

**What the tails tell us.** A small number of funds show economically impossible
values — slightly negative drag, or implausibly large gaps. These come from NAV
data artefacts (a stale or back-filled price near a window boundary, or a plan
whose effective history is barely over our 2-year floor). They are **noise, not
signal**.

**Decision:** rather than delete these funds (which biases the sample) or let them
distort means, we keep them but **report the median** as our headline statistic
and **winsorise at the 1st/99th percentile** only for the regression in nb 04
(where extreme leverage points would otherwise dominate). The median is robust to
these tails by construction, so our central finding doesn't depend on this choice.
We flag the affected rows transparently rather than hiding them.

In [7]:
lo, hi = df["drag_pp"].quantile([0.01, 0.99])
df["drag_pp_winsor"] = df["drag_pp"].clip(lo, hi)
df["is_outlier_flag"] = (df["drag_pp"] < 0) | (df["drag_pp"].abs() > 4)
print(f"Winsorised drag to [{lo:.2f}, {hi:.2f}] pp/yr for modelling.")
print(f"Flagged {df['is_outlier_flag'].sum()} rows as data-quality outliers (kept, not dropped).")
print(f"\nHeadline median drag (robust): {df['drag_pp'].median():.2f} pp/yr")
print(f"Median drag, Equity only:      {df.loc[df.broad_category=='Equity','drag_pp'].median():.2f} pp/yr")

Winsorised drag to [0.00, 2.19] pp/yr for modelling.
Flagged 5 rows as data-quality outliers (kept, not dropped).

Headline median drag (robust): 0.81 pp/yr
Median drag, Equity only:      1.44 pp/yr


## Step 6 — Save the analysis-ready dataset

One tidy row per fund. This is the single input to notebooks 03 and 04, so those
notebooks never touch raw JSON — clean separation between *building* the dataset
and *analysing* it.

In [8]:
cols = ["fund_name", "fund_house", "broad_category", "scheme_category",
        "direct_code", "regular_code", "window_start", "window_end", "window_years",
        "direct_cagr", "regular_cagr", "drag_pp", "drag_pp_winsor", "is_outlier_flag"]
df_out = df[cols].sort_values("drag_pp", ascending=False).reset_index(drop=True)
paths.PROCESSED.mkdir(parents=True, exist_ok=True)
df_out.to_csv(paths.RETURNS_CSV, index=False)
print(f"Wrote {len(df_out):,} funds -> {paths.RETURNS_CSV}")
df_out.head()

Wrote 1,084 funds -> /Users/shauryakesarwani/Desktop/prjcts/hidden-mutual-fund-fees/data/processed/fund_returns.csv


,fund_name,fund_house,broad_category,scheme_category,direct_code,regular_code,window_start,window_end,window_years,direct_cagr,regular_cagr,drag_pp,drag_pp_winsor,is_outlier_flag
0,Principal Small Cap Fund,Principal Mutual Fund,Equity,Equity Scheme - Small Cap Fund,147131,147129,2019-05-14,2021-12-30,2.63,0.435458,0.410290,2.516718,2.185423,False
1,Sundaram Equity Fund,Sundaram Mutual Fund,Equity,Equity Scheme - Multi Cap Fund,147587,147584,2019-09-11,2021-12-31,2.31,0.268327,0.245050,2.327687,2.185423,False
2,ITI Mid Cap Fund,ITI Mutual Fund,Equity,Equity Scheme - Mid Cap Fund,148733,148732,2021-03-10,2026-06-05,5.24,0.185532,0.162665,2.286648,2.185423,False
3,ITI ELSS Tax Saver Fund,ITI Mutual Fund,Equity,Equity Scheme - ELSS,147541,147544,2019-10-25,2026-06-05,6.61,0.161038,0.138362,2.267601,2.185423,False
4,ITI Pharma and Healthcare Fund,ITI Mutual Fund,Equity,Equity Scheme - Sectoral/ Thematic,149268,149270,2021-11-11,2026-06-05,4.56,0.138933,0.116363,2.256932,2.185423,False


### Cleaning summary (what we did and why)

| Step | Decision | Why |
|---|---|---|
| Plans | Keep **Growth** only, drop IDCW/dividend | Dividend payouts distort NAV-based returns |
| Pairing | Require exactly one Direct & one Regular per fund | Avoids mis-matching similarly-named schemes |
| Window | Compare over the **common** date range only | Direct plans start in 2013; a fair race needs the same start/finish |
| Min history | Require **≥ 2 years** overlap | Short windows are dominated by tracking noise, not fees |
| Scheme type | Keep **open-ended** only | FMPs/close-ended series aren't buy-and-hold investments |
| Non-positive / dup NAVs | Dropped in `nav_payload_to_series` | Data errors |
| Outliers | **Flag + winsorise (1/99) for modelling**, report **median** headline | Robust to impossible/extreme artefacts without dropping funds |

**Next:** `03_analysis` answers the four headline questions on this clean dataset.